In [1]:
# Name: Muhammad Salman
# Student ID: IS01083889
# CISB5123 Text Analytics - Lab Assignment 1: Web Scraping
# Platform: Shopee Malaysia (Google Play Store Reviews)

!pip install google-play-scraper pandas

import pandas as pd
import time
from google_play_scraper import reviews, Sort


def scrape_shopee_reviews(app_id, pages=5, reviews_per_page=40):
    """
    Scrapes Shopee Malaysia app reviews from the Google Play Store.
    Uses a continuation_token to paginate across multiple pages.

    Parameters:
        app_id (str): Google Play package name for the app
        pages (int): Number of pages to scrape (default: 5)
        reviews_per_page (int): Reviews per page (default: 40)

    Returns:
        list: Raw review dictionaries from the API
    """
    all_reviews = []
    continuation_token = None

    for page in range(1, pages + 1):
        print(f"Scraping page {page} of {pages}...")

        # Fetch reviews using Play Store API, newest first
        result, continuation_token = reviews(
            app_id,
            lang='en',
            country='my',              # Malaysia store
            sort=Sort.NEWEST,          # Newest reviews first
            count=reviews_per_page,
            continuation_token=continuation_token  # Pagination cursor
        )

        all_reviews.extend(result)
        time.sleep(1)  # Polite delay between requests

        if not continuation_token:
            print("No more pages available.")
            break

    print(f"\nTotal reviews scraped: {len(all_reviews)}")
    return all_reviews


def extract_review_fields(raw_reviews):
    """
    Extracts only the 3 required fields from raw Play Store review data.

    Parameters:
        raw_reviews (list): Raw review dicts from scrape_shopee_reviews()

    Returns:
        list: Cleaned dicts with reviewer_name, review_date, review_content
    """
    cleaned = []

    for review in raw_reviews:
        cleaned.append({
            'reviewer_name':  review.get('userName', 'N/A'),   # Reviewer display name
            'review_date':    str(review.get('at', 'N/A')),    # Date review was posted
            'review_content': review.get('content', 'N/A')     # Review text
        })

    return cleaned


def save_to_csv(data, filename='shopee_reviews.csv'):
    """
    Saves cleaned review data to a CSV file using pandas.
    Uses utf-8-sig so special characters display correctly in Excel.

    Parameters:
        data (list): Cleaned review dicts from extract_review_fields()
        filename (str): Output CSV filename

    Returns:
        pd.DataFrame: The saved DataFrame
    """
    df = pd.DataFrame(data)
    df.to_csv(filename, index=False, encoding='utf-8-sig')
    print(f"Saved {len(df)} reviews to '{filename}'")
    return df


# ── Main Execution ────────────────────────────────────────────────────────────
SHOPEE_APP_ID = 'com.shopee.my'  # Shopee Malaysia package name on Google Play

# Step 1: Scrape 5 pages of reviews
raw_reviews = scrape_shopee_reviews(SHOPEE_APP_ID, pages=5, reviews_per_page=40)

# Step 2: Extract the 3 required fields
cleaned_reviews = extract_review_fields(raw_reviews)

# Step 3: Save to CSV
df = save_to_csv(cleaned_reviews, filename='shopee_reviews.csv')

# Step 4: Preview
print(f"\nShape: {df.shape}")
df.head(10)

Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels
Scraping page 1 of 5...
Scraping page 2 of 5...
Scraping page 3 of 5...
Scraping page 4 of 5...
Scraping page 5 of 5...

Total reviews scraped: 200
Saved 200 reviews to 'shopee_reviews.csv'

Shape: (200, 3)


,reviewer_name,review_date,review_content
0,H4V0C5,2026-03-01 14:47:20,bad service refund
1,Rana Munawar,2026-03-01 14:35:35,Nice app
2,Amin Adnan,2026-03-01 12:33:19,ok
3,Amir Husin,2026-03-01 12:02:02,"Mantap,barang lebih murah dari online shopping..."
4,Sarah jazzera,2026-03-01 11:51:05,why? you become more like tiktok ban here ban ...
5,Isya MJ,2026-03-01 11:22:06,these few days my apps is not working as usual...
6,Neo,2026-03-01 10:53:58,Not receive refund and no help provided
7,sam,2026-03-01 10:50:11,⭐️⭐️⭐️⭐️⭐️⭐️⭐️⭐️⭐️⭐️⭐️⭐️⭐️⭐️⭐️⭐️
8,Shukri Zainudain,2026-03-01 10:30:05,"The app is intrusive, it ran itself in the bac..."
9,Last Samurai,2026-03-01 10:18:57,terbaik
